In [ ]:
import pandas as pd
import numpy as np
import os


In [ ]:
DIALECTS = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"]
SEEDS = [0, 1, 2, 3, 4]

PREFIX_PATH = './openai_moderation_typo_results'
DATA_TYPE = 'benign'

final_stats = []

print("📊 [Typo Moderation] SAE+Typo vs Dialect comparison...\n")

for dialect in DIALECTS:
    tpr_sae_list = []
    tpr_dialect_list = []
    tpr_sae_typo_list = []
    tpr_gap_list = []
    
    type_cos_sim_list = []
    
    if DATA_TYPE == 'toxic':
        std_dial_path = f'./openai_moderation_results/{dialect}_toxic.csv'
    else:
        std_dial_path = f'./openai_moderation_results/{dialect}_benign.csv'
        
    if not os.path.exists(std_dial_path):
        print(f"  ⚠ [Warning] Base file missing. Skipping {dialect}!")
        continue
        
    std_dial_df = pd.read_csv(std_dial_path)[['standard_is_harmful', 'dialect_is_harmful']]
    
    for seed in SEEDS:
        csv_path = f"{PREFIX_PATH}/{dialect}_{DATA_TYPE}_results_with_typo_seed_{seed}.csv"
        
        dataset_file = f"./text_level_typo_results/{DATA_TYPE}_results_{dialect}_seed_{seed}.csv"
        
        if not os.path.exists(csv_path):
            continue
            
        df_typo = pd.read_csv(csv_path)
        df_dataset = pd.read_csv(dataset_file)
        
        if len(df_typo) != len(std_dial_df):
            print(f"  ❌ [Error] {dialect} seed {seed} row count mismatch! Skipping.")
            continue
            
        df = pd.concat([df_typo, std_dial_df], axis=1)
        total_samples = len(df)
        
        if total_samples == 0:
            continue
        
        type_cos_sim_list.append(df_dataset['typo_sim'].mean())
        
        if DATA_TYPE == 'toxic':
            std_nsfw_blocked = df['standard_is_harmful'].sum()
            dialect_nsfw_blocked = df['dialect_is_harmful'].sum()
            sae_typo_nsfw_blocked = df['typo_is_harmful'].sum()
            
            std_tpr = (std_nsfw_blocked / total_samples) * 100
            dialect_tpr = (dialect_nsfw_blocked / total_samples) * 100
            sae_typo_tpr = (sae_typo_nsfw_blocked / total_samples) * 100
            
            tpr_gap = dialect_tpr - sae_typo_tpr
            
            tpr_sae_list.append(std_tpr)
            tpr_dialect_list.append(dialect_tpr)
            tpr_sae_typo_list.append(sae_typo_tpr)
            tpr_gap_list.append(tpr_gap)
        else:
            # For benign data, is_harmful == True indicates a false positive (FP)
            std_fp_count = df['standard_is_harmful'].sum()
            dialect_fp_count = df['dialect_is_harmful'].sum()
            sae_typo_fp_count = df['typo_is_harmful'].sum() 
            
            std_fpr = (std_fp_count / total_samples) * 100
            dialect_fpr = (dialect_fp_count / total_samples) * 100
            sae_typo_fpr = (sae_typo_fp_count / total_samples) * 100
            
            fpr_gap = dialect_fpr - sae_typo_fpr
            
            tpr_sae_list.append(std_fpr)
            tpr_dialect_list.append(dialect_fpr)
            tpr_sae_typo_list.append(sae_typo_fpr)
            tpr_gap_list.append(fpr_gap)
            
    if tpr_sae_list:
        avg_tpr_sae = np.mean(tpr_sae_list)
        avg_tpr_dialect = np.mean(tpr_dialect_list)
        
        avg_tpr_sae_typo = np.mean(tpr_sae_typo_list)
        std_tpr_sae_typo = np.std(tpr_sae_typo_list, ddof=1)
        
        avg_gap = np.mean(tpr_gap_list)
        std_gap = np.std(tpr_gap_list, ddof=1)
        
        
        avg_typo_cos_sim = np.mean(type_cos_sim_list)
        std_typo_cos_sim = np.std(type_cos_sim_list, ddof=1)
        
        print(f"Dialect: {dialect} | cos sim mean: {avg_typo_cos_sim}, cos sim std.:{std_typo_cos_sim}")
        
        if DATA_TYPE == 'toxic':
        
            final_stats.append({
                'Dialect': dialect,
                'TPR_SAE (%)': round(avg_tpr_sae, 3),
                'TPR_Dialect (%)': round(avg_tpr_dialect, 3),
                'TPR_SAE+Typo_Mean (%)': round(avg_tpr_sae_typo, 3),
                'TPR_SAE+Typo_Std': round(std_tpr_sae_typo, 3),
                'Gap (Dialect - Typo) (%p)': round(avg_gap, 3),
                'Gap Std': round(std_gap, 3),
                'SAD <-> Typo cos sim. Mean': round(avg_typo_cos_sim, 3),
                'SAD <-> Typo cos sim. Std. ': round(std_typo_cos_sim, 3),
            })
        else:
            final_stats.append({
                'Dialect': dialect,
                'FPR_SAE (%)': round(avg_tpr_sae, 3),
                'FPR_Dialect (%)': round(avg_tpr_dialect, 3),
                'FPR_SAE+Typo_Mean (%)': round(avg_tpr_sae_typo, 3),
                'FPR_SAE+Typo_Std': round(std_tpr_sae_typo, 3),
                'Gap (Dialect - Typo) (%p)': round(avg_gap, 3),
                'Gap Std': round(std_gap, 3),
                'SAD <-> Typo cos sim. Mean': round(avg_typo_cos_sim, 3),
                'SAD <-> Typo cos sim. Std. ': round(std_typo_cos_sim, 3),
            })

summary_df = pd.DataFrame(final_stats)

print("✅ Summary DataFrame generated!\n")
print(summary_df.to_string(index=False))

In [ ]:
summary_df.keys()

In [ ]:
try:
    print(summary_df[['Dialect', 'TPR_SAE (%)', 'TPR_SAE+Typo_Mean (%)', 'TPR_SAE+Typo_Std', 'Gap (Dialect - Typo) (%p)', 'Gap Std']])
except:
    print(summary_df[['Dialect', 'FPR_SAE (%)', 'FPR_SAE+Typo_Mean (%)', 'FPR_SAE+Typo_Std', 'Gap (Dialect - Typo) (%p)', 'Gap Std']])

In [ ]:
summary_df[['SAD <-> Typo cos sim. Mean', 'SAD <-> Typo cos sim. Std. ']]